In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
from tqdm import tqdm

from typing import Literal
from scipy.signal import convolve

import sys, os
sys.path.append(os.path.join(os.getcwd(), ".."))


np.random.seed(1)
torch.manual_seed(1)
torch.cuda.manual_seed(1)

from ml_force import MorrisLecar, MorrisLecarCurrent, z_transform, minmax_transform

In [ ]:
save_dir = os.path.join(os.getcwd(), "memory_task", "figures")
os.makedirs(save_dir, exist_ok=True)

print(save_dir)

## Create Supervisor as a memory task

Random duration of several images

In [ ]:
multiplier = 1
n_images = 10
width, height = 16, 9

images = np.array([np.random.choice([-1, 1], size=(height, width), replace=True) for _ in range(n_images)]) * multiplier

In [ ]:
# Test to see if ravel() and reshape() are inverses

ind = 2
print(np.sum(images[ind] == images[ind].ravel().reshape(height, width)) == width * height)

In [ ]:
for i in range(n_images):
    for j in range(i+1, n_images):
        if np.sum(images[i] == images[j]) == width * height:
            print(i, j)

In [ ]:
def unravel_image(image, height:int, width:int):
    return image.ravel().reshape(height, width)

In [ ]:
print(images[2])

In [ ]:
fig, ax = plt.subplots(figsize=(2 * n_images, 5 * n_images), ncols=n_images)

for i in range(len(ax)):
    ax[i].imshow(images[i])
    ax[i].set_title(f"Image {i+1}")
    ax[i].set_xticks([])
    ax[i].set_yticks([])
plt.show()

In [ ]:
dt = 0.05   # ms
n_exposures = 30     # number of images shown
nt_min = 2000 // dt
nt_max = 4000 // dt

t_transient = 1000  # ms
nt_transient = int(t_transient // dt)

signal = []
exposures = []
for i in range(n_images):
    signal += [np.zeros(height * width) for _ in range(nt_transient)]   # show a blank screen for `t_transient` ms
    nt = np.random.randint(nt_min, nt_max)  # number of time steps to show the image
    print(round(nt * dt, ndigits=2))
    image_index = np.random.randint(0, n_images)
    exposures.append(image_index)       # save the image index for each exposure
    image_raveled = images[image_index].ravel()   # ravel the image
    signal += [image_raveled for _ in range(nt)]    # show the image for nt time steps

signal = np.array(signal)
print(f"Total signal length: {signal.shape[0]} --> {(signal.shape[0] * dt):.2f} ms")
print(f"Exposures: {np.unique(exposures)}")

In [ ]:
def smooth(signal, window_len:int=5, window:Literal['hanning', 'hamming', 'bartlett', 'blackman']='flat', 
           mode:Literal['full', 'valid', 'same']='same', axis:int=0, 
           method:Literal['auto', 'direct', 'fft']='auto'):
    """
    Smooth the signal using a window with requested size.
    This method is based on the convolution of a scaled window with the signal.
    The signal is prepared by introducing reflected copies of the signal (with the window size) in both ends
    so that transient parts are minimized in the begining and end part of the output signal.
    """
    if signal.ndim != 2:
        raise ValueError("Signal must be 2D")
    if window_len < 3:
        return signal
    if signal.shape[axis] < window_len:
        raise ValueError("Input signal must be longer than window size")
    if window == 'flat':
        kernel = np.ones(window_len) / window_len
    else:
        kernel = eval(f"np.{window}(window_len)")
    if axis==0:
        kernel_nd = kernel[:, np.newaxis]
    elif axis==1:
        kernel_nd = kernel[np.newaxis, :]
    else:
        raise ValueError("Axis must be 0 or 1")
    return convolve(signal, kernel_nd / kernel_nd.sum(), mode=mode, method=method)

In [ ]:
window_len = 300 // dt
signal_smoothed = smooth(signal, window_len=window_len, window='hanning', mode="same", method='fft', axis=0)

In [ ]:
t_train = np.arange(signal.shape[0]) * dt

pixel = 5
plt.plot(t_train, signal[:, pixel], label="Original")
plt.plot(t_train, signal_smoothed[:, pixel], label="Smoothed")
plt.legend()
plt.show()

## Instantiate the ML class

In [ ]:
N = 800
T = signal_smoothed.shape[0] * dt
I_bias = 75      # mA

Q = 100
gbar = 15   # nS
lamda = 0.8
p_sparsity = 0.1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

ml = MorrisLecar(supervisor=signal_smoothed, N=N, T=T, dt=dt, BIAS=I_bias, Q=Q, gbar=gbar, 
                #  p_sparsity=p_sparsity,
                 l=lamda, device=device)

## Training

In [ ]:
def save_reservoir_state(ml:MorrisLecar, save_prefix:str=None):
    """Save the reservoir state, including n, s, eta, and dec.
    Also save the images and exposures shown to the reservoir.
    """
    if save_prefix is None:
        save_prefix = ""
        
    save_dir = os.path.join(os.getcwd(), "memory_task", "reservoir_states" + save_prefix)
    os.makedirs(save_dir, exist_ok=True)
    
    np.save(os.path.join(save_dir, "n.npy"), ml.n.cpu().numpy())
    np.save(os.path.join(save_dir, "s.npy"), ml.s.cpu().numpy())
    np.save(os.path.join(save_dir, "eta.npy"), ml.eta.cpu().numpy())
    np.save(os.path.join(save_dir, "dec.npy"), ml.dec.cpu().numpy())
    np.save(os.path.join(save_dir, "images.npy"), images)
    np.save(os.path.join(save_dir, "exposures.npy"), exposures)
    
    return

In [ ]:
def load_model(load_prefix:str=None):
    if load_prefix is None:
        load_prefix = ""
        
    save_dir = os.path.join(os.getcwd(), "memory_task", "reservoir_states" + load_prefix)
    
    n = torch.from_numpy(np.load(os.path.join(save_dir, "n.npy"))).to(device)
    s = torch.from_numpy(np.load(os.path.join(save_dir, "s.npy"))).to(device)
    eta = torch.from_numpy(np.load(os.path.join(save_dir, "eta.npy"))).to(device)
    dec = torch.from_numpy(np.load(os.path.join(save_dir, "dec.npy"))).to(device)
    images = np.load(os.path.join(save_dir, "images.npy"))
    exposures = np.load(os.path.join(save_dir, "exposures.npy"))
    
    return n, s, eta, dec, images, exposures

In [ ]:
nt = ml.sup.shape[0]

transient_time = 500
nt_transient = int(transient_time // dt)

print("transient time:", transient_time)
for i in tqdm(range(nt_transient)):
    ml.euler_step(closed_loop=False)

print(f"Training Time: {round(ml.sup.shape[0]*ml._dt, ndigits=2)} ms")
for i in tqdm(range(nt)):
    ml.euler_step(closed_loop=True, voltage_bound=None)
    ml.x_hat_rec[i] = ml.x_hat.ravel()
    
    if i % 20 == 1:
        ml.rls(i)

In [ ]:
save_reservoir_state(ml, save_prefix=f"_N{N}_Q{Q}_gbar{gbar}_l{lamda}_p{p_sparsity}")

In [ ]:
ml.x_hat_rec.shape, ml.sup.shape

In [ ]:
nrows = 10
fig, ax = plt.subplots(figsize=(5, nrows * 1), nrows=nrows, sharex=True, sharey=True)
plt.ylim(-2, 2)

for i in range(len(ax)):
    ax[i].plot(t_train, ml.sup[:, i].cpu().numpy(), label="supervisor")
    ax[i].plot(t_train, ml.x_hat_rec[:, i].cpu().numpy(), '--', label="output")
    ax[i].set_ylabel(f"Pixel {i+1}")
ax[-1].set_xlabel("Time (ms)")
plt.legend()
plt.show()

## Test

1. Choose an image as target
2. Add Gaussian noise to it
3. Bring the system to the noisy target
4. Set the bias current to baseline and close the feedback loop and watch what happens

In [ ]:
image_id = exposures[1]

target_image = images[image_id].ravel().reshape(-1, 1)
noise = np.random.normal(0, 0.1, target_image.shape)
noisy_target_image = target_image + noise

fig, ax = plt.subplots(figsize=(10, 5), ncols=2)

vmin = min(target_image.min(), noisy_target_image.min())
vmax = max(target_image.max(), noisy_target_image.max())

clr1 = ax[0].imshow(target_image.reshape(height, width), vmin=vmin, vmax=vmax)
ax[0].set_title("Target Image")
clr2 = ax[1].imshow(noisy_target_image.reshape(height, width), vmin=vmin, vmax=vmax)
ax[1].set_title("Noisy Image")
plt.colorbar(clr1, ax=ax[0])
plt.colorbar(clr2, ax=ax[1])
plt.show()

### Test by exposing the model to a noisy image

In [ ]:
duration = 2000
nt_transient = int(duration // dt)

ml._BIAS = I_bias + ml.eta @ torch.tensor(noisy_target_image, dtype=torch.float32, device=device)
s_rec = torch.zeros((nt_transient, ml._N), dtype=torch.float32, device=device)  # recording the network state (i.e., synaptic-gating variables)
print("Bringing the network to the new equilibrium...")
for i in tqdm(range(nt_transient)):
    ml.euler_step(closed_loop=False)
    s_rec[i] = ml.s.ravel()

print("Started Testing...")
nt_test = int(duration // dt)
ml._BIAS = I_bias   # reset the bias to the original value
output = torch.zeros((nt_test, target_image.shape[0]), device=device)   # recording the output
for i in tqdm(range(nt_test)):
    ml.euler_step(closed_loop=True, voltage_bound=200)
    output[i] = ml.x_hat.ravel()

### Test by initializing s and v near the equilibrium

In [ ]:
duration = 2000
nt_transient = int(duration // dt)

print(f"Transient Period: {duration} ms")   # transient period
for i in tqdm(range(nt_transient)):
    ml.euler_step(closed_loop=False)

In [ ]:
s_init = torch.linalg.pinv(ml.dec.T) @ torch.tensor(noisy_target_image, dtype=torch.float32, device=device)
ml.s = s_init.clone().detach()
s_rec = torch.zeros((nt_transient, ml._N), dtype=torch.float32, device=device)  # recording the network state

print("Started Testing...")
nt_test = int(duration // dt)
output = torch.zeros((nt_test, target_image.shape[0]), device=device)   # recording the output
for i in tqdm(range(nt_test)):
    ml.euler_step(closed_loop=True, voltage_bound=None)
    output[i] = ml.x_hat.ravel()
    s_rec[i] = ml.s.ravel()

In [ ]:
neurons = np.random.randint(0, ml._N, 5)
print(neurons)

fig, ax = plt.subplots(figsize=(5, 10), nrows=neurons.shape[0], sharex=True, sharey=True)
for i in range(neurons.shape[0]):
    ax[i].plot(np.arange(nt_transient) * dt, s_rec[:, neurons[i]].cpu().numpy())
    ax[i].set_title(f"Neuron {neurons[i]}")
plt.show()

In [ ]:
t_test = np.arange(nt_test) * dt

nrows = target_image.shape[0]
fig, ax = plt.subplots(figsize=(5, nrows*1), nrows=nrows, sharex=True)
plt.suptitle("Results for the Memory Task", y=0.90)
for i in range(len(ax)):
    ax[i].plot(t_test, noisy_target_image[i] * np.ones(nt_test), label="noisy target")
    ax[i].plot(t_test, target_image[i] * np.ones(nt_test), label="target")
    ax[i].plot(t_test, output[:, i].cpu().numpy(), label="output")
    ax[i].set_ylabel(f"Pixel {i+1}")
ax[-1].set_xlabel("Time (ms)")
ax[0].legend(loc='upper right')
plt.savefig(os.path.join(save_dir, "output_series.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
leng = output.shape[0]
model_render = unravel_image(output[-2000:].cpu().numpy().mean(axis=0), height, width)


vmin = min(noisy_target_image.min(), target_image.min(), model_render.min())
vmax = max(noisy_target_image.max(), target_image.max(), model_render.max())

fig, ax = plt.subplots(figsize=(15, 5), ncols=3)
clr1 = ax[0].imshow(unravel_image(noisy_target_image, height, width), vmin=vmin, vmax=vmax)
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].set_title("Noisy Image")
clr2 = ax[1].imshow(unravel_image(target_image, height, width), vmin=vmin, vmax=vmax)
ax[1].set_xticks([])
ax[1].set_yticks([])
ax[1].set_title("Target Image")
clr3 = ax[2].imshow(model_render, vmin=vmin, vmax=vmax)
ax[2].set_xticks([])
ax[2].set_yticks([])
ax[2].set_title("Model Render")
plt.colorbar(clr1, ax=ax[0])
plt.colorbar(clr2, ax=ax[1])
plt.colorbar(clr3, ax=ax[2])
plt.suptitle("Memory Task Image Comparison", y=0.97, fontsize=16)
plt.savefig(os.path.join(save_dir, "image_comparison.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for i in range(n_images):
    plt.imshow(images[i])
    plt.title(f"Image {i+1}")
    plt.xticks([])
    plt.yticks([])
    plt.show()
    plt.close()

In [ ]:
exposures[-1]

### Large-scale test

In [ ]:
# n_tasks = 6
n_tests = 50

test_images = np.array([images[np.random.choice(exposures, size=1, replace=True)].ravel().reshape(-1, 1) for _ in range(n_tests)])
corrupted_test_images = np.array([test_images[i] + np.random.normal(0, 0.1, test_images[i].shape) for i in range(n_tests)])

duration = 1000
nt_transient = int(duration // dt)
nt_test = int(duration // dt)


renders = []
for i in range(n_tests):
    print(f"Testing Image {i+1}/{n_tests}")
    print(f"Transient Period: {duration} ms")   # transient period
    for _ in tqdm(range(nt_transient)):
        ml.euler_step(closed_loop=False)
    
    # Calculate the initial state for the network
    s_init = torch.linalg.pinv(ml.dec.T) @ torch.tensor(corrupted_test_images[i], dtype=torch.float32, device=device)
    ml.s = s_init.clone().detach()      # reset the network state
    # s_rec = torch.zeros((nt_transient, ml._N), dtype=torch.float32, device=device)  # recording the network state

    print(f"Testing Period: {duration} ms")
    output = torch.zeros((nt_test, test_images[i].shape[0]), device=device)   # recording the output
    for i in tqdm(range(nt_test)):
        ml.euler_step(closed_loop=True, voltage_bound=150)
        output[i] = ml.x_hat.ravel()
        # s_rec[i] = ml.s.ravel()
    model_render = output[-2000:].cpu().numpy().mean(axis=0)
    renders.append(model_render)

In [ ]:
losses = [np.sqrt(np.sum((test_images[i] - renders[i])**2)) for i in range(len(renders))]
print(f"Average Loss: {np.mean(losses):.4f}")


In [ ]:
vmin = min(corrupted_test_images.min(), test_images.min(), np.min(renders))
vmax = max(corrupted_test_images.max(), test_images.max(), np.max(renders))

for i in range(len(renders)):
    plt.imshow(unravel_image(renders[i], height, width), vmin=vmin, vmax=vmax)
    plt.xticks([])
    plt.yticks([])
    plt.colorbar()
    plt.title(f"Loss: {losses[i]:.4f}")
    plt.show()
    plt.close()